# Week 23: SAR: Sentinel-1, polarimetry, InSAR

**Track:** Space GIS Architect (Expert)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/23/](https://launchdetect.com/academy/week/23/)  
**Track index:** [https://launchdetect.com/academy/space-gis-architect/](https://launchdetect.com/academy/space-gis-architect/)

---

_SAR sees through clouds, day and night. It can measure ground deformation to the centimeter. This week you decode the magic._


## Why this week matters

SAR sees through clouds and works day-night. For any application where optical is unreliable (tropical regions, monsoons, polar night), SAR is the only option. InSAR adds ground-deformation measurement to mm precision.


## Learning objectives

By the end of this lab you will be able to:

- Understand synthetic aperture radar fundamentals
- Distinguish single, dual, and quad polarization
- Compute interferometric phase between two SAR acquisitions (InSAR)
- Detect ground deformation at centimeter scale with InSAR


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio pystac-client boto3`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio pystac-client boto3


## The approach (and why)

Simulate a synthetic deformation signal to understand how it appears in an interferogram. Real Sentinel-1 SLC processing requires snap or pyrosar — both heavyweight installs better done on a dedicated workstation than Colab.


In [ ]:
# Week 23: SAR / InSAR concepts — synthetic interferogram demo.
import numpy as np, matplotlib.pyplot as plt

# Simulate a circular deformation pattern (volcanic uplift, ~10 cm peak)
x, y = np.meshgrid(np.linspace(-1, 1, 100), np.linspace(-1, 1, 100))
r = np.sqrt(x**2 + y**2)
deformation_cm = 10 * np.exp(-r**2 * 5)

# Sentinel-1 C-band wavelength (5.6 cm). Phase wraps every half-wavelength.
lambda_cm = 5.6
phase_rad = (4 * np.pi / lambda_cm) * deformation_cm
wrapped_phase = np.angle(np.exp(1j * phase_rad))  # wrap to [-π, π]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(deformation_cm, cmap="RdBu_r"); axes[0].set_title("True deformation (cm)")
axes[1].imshow(wrapped_phase, cmap="hsv"); axes[1].set_title("Wrapped interferogram phase")
plt.tight_layout(); plt.show()

# TODO: download two real Sentinel-1 SLC scenes over a deformation event
# from ASF DAAC, coregister, form the interferogram with snap or pyrosar.


## What just happened — and why it works

InSAR phase wraps every half-wavelength of slant-range motion. For Sentinel-1's 5.6 cm C-band, that's 2.8 cm of vertical deformation per fringe. Unwrapping the phase recovers the absolute deformation field.


## Common gotchas

- Coherence depends on surface stability between acquisitions. Vegetation, snow, and water are decoherent — phase is noise there.
- Atmospheric water vapor introduces phase delay that masquerades as deformation. Use tropospheric models (GACOS, ERA5) to correct.
- Phase unwrapping is the hardest step. SNAPHU is the standard tool but has known edge cases.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/23/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
